### Preprocessing and features engineering

Now we have a clean dataset with tennis matches starting from **2005-07-04** until **2026** and we need to preprocess it, by reducing the number of features and creating new ones to improve the model performance. Afterwards, we will split the dataset into training and test sets and start the model training.

In [5]:
import pandas as pd
import numpy as np

df_raw = pd.read_excel("../data/raw/tennis_matches_combined.xlsx")
df_raw.head()

,Date,Series,Court,Surface,Round,Best of,Winner,Loser,WRank,LRank,...,W2,L2,W3,L3,W4,L4,W5,L5,B365W,B365L
0,2005-07-04,International,Outdoor,Clay,1st Round,3.0,Robredo T.,Tabara M.,20.0,112.0,...,6.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1.10,6.00
1,2005-07-04,International,Outdoor,Clay,1st Round,3.0,Vinciguerra A.,Ryderstedt M.,917.0,132.0,...,6.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2.10,1.66
2,2005-07-05,International,Outdoor,Clay,1st Round,3.0,Youzhny M.,Haehnel J.,27.0,109.0,...,6.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.19,4.00
3,2005-07-05,International,Outdoor,Clay,1st Round,3.0,Ferrero J.C.,Dlouhy L.,31.0,136.0,...,6.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1.07,7.00
4,2005-07-05,International,Outdoor,Clay,1st Round,3.0,Berdych T.,Kim K.,42.0,71.0,...,6.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,1.19,4.00


#### Numerical features

We can start by creating simple engineered features, such as the differences between the players' ranks, points, and betting odds.

We can also use logarithmic differences. This is useful because these variables are not linear.

**Example 1**

- Rank 1 vs 10
- Rank 101 vs 110

The absolute difference is the same (9), but the first difference is much more meaningful than the second one.

**Example 2**

- Points 9000 vs 8000
- Points 1200 vs 1000

Same considerations as example 1.

**Example 3**

- Odds 1.20 vs 1.50
- Odds 4.00 vs 4.30

Same absolute difference, but the second one is much more uncertain than the first one.

We also apply a **Laplace correction** to betting odds, to avoid computing the logarithm of zero.

In [8]:
df_prepared = df_raw.copy()

wRank, lRank = df_raw["WRank"], df_raw["LRank"]
wPoints, lPoints = df_raw["WPts"], df_raw["LPts"]
wOdds, lOdds = df_raw["B365W"], df_raw["B365L"]

df_prepared["Log_Rank_Diff"] = np.log(wRank) - np.log(lRank)
df_prepared["Log_Points_Diff"] = np.log(wPoints) - np.log(lPoints)
df_prepared["Log_Odds_Diff"] = np.log(wOdds+0.1) - np.log(lOdds+0.1)

We can convert **Best of** feature into a binary feature **Best of 5**, which is 1 if the match is best of 5 sets and 0 otherwise.

In [9]:
df_prepared["Best_of_5"] = (df_raw["Best of"] == 5).astype(int)

Finally, we can create an **imputer** to handle missing values in the dataset. We will use this later, after the splitting phase, to avoid data leakage.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numerical_features = [
    "Log_Rank_Diff",
    "Log_Points_Diff",
    "Log_Odds_Diff",
    "Best_of_5"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    ))
])

#### Categorical features

We create an imputer to manage missing values, using a **most frequent** strategy and **One hot encoding** to convert categorical features into numerical ones.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "Series",
    "Court",
    "Surface",
    "Round"
]

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# Create a preprocessor that combines both numeric and categorical pipelines
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features),
])

#### Advanced features

We can improve the model performance by creating *advanced* features about players ELO ratings, fatigue and head to head statistics.